# Lab 7 · Chuỗi & regex trên 690 nghìn review

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành tuần 7**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo buổi 7 xử lý cột `name` và `amenities` của bảng listings. Lab này bước
sang cột văn bản lớn nhất của cả bộ dữ liệu: **comments** của bảng reviews đầy đủ —
nơi bạn sẽ dựng các tín hiệu chuẩn bị trực tiếp cho hợp phần LLM (buổi 11).

## Cách làm việc trong buổi lab

- Bài tập được chia bước; mỗi bước có ô `TODO` và phần kiểm tra `assert` — chạy qua hết
  `assert` nghĩa là bạn làm đúng.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🚫 ở giờ lý thuyết đo đúng các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn ✅ mở: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Bạn kẹt quá 3 phút ở một bước: gọi trợ giảng.

## Mục tiêu

Sau buổi lab, bạn:

1. Khảo sát một cột văn bản lớn: độ dài, giá trị rỗng, "rác" đặc trưng.
2. Làm sạch HTML thừa bằng `str.replace` và kiểm chứng bằng đếm.
3. Viết heuristic ngôn ngữ và tín hiệu từ khoá bằng `str.contains`.
4. Dùng `str.extract` trên văn bản tự do — và tự soi false positive của chính mình.

## Phần 0 · Khởi động (~8 phút)

In [ ]:
import pandas as pd

# W1 — chuẩn hoá trước khi đếm
s = pd.Series(["  Wifi ", "wifi", "WIFI!", "wi-fi"])

# TODO: strip + lower cho cả cột
s_chuan = ...

# --- Ô kiểm tra ---
assert list(s_chuan) == ["wifi", "wifi", "wifi!", "wi-fi"]
assert (s_chuan == "wifi").sum() == 2
print("W1 ổn — chuẩn hoá gộp được 2 biến thể; 2 biến thể còn lại cần thêm regex.")

In [ ]:
# W2 — bẫy NaN của contains
t = pd.Series(["depto centro", None, "casa con vista"])

# TODO: đếm số dòng chứa "centro" sao cho dòng None được tính là KHÔNG chứa
so_chua = ...

# --- Ô kiểm tra ---
assert so_chua == 1
print("W2 ổn — na=False biến 'không biết' thành 'không chứa' một cách CÓ CHỦ ĐÍCH.")

## Phần 1 · Khảo sát cột văn bản lớn (~25 phút)

Bảng reviews **đầy đủ** có cột `comments` — 690 nghìn đoạn văn khách viết, đủ thứ tiếng.

In [ ]:
URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/data/reviews.csv.gz")
rv = pd.read_csv(URL, usecols=["listing_id", "date", "comments"],
                 parse_dates=["date"])
len(rv)

### Bước 1 · Nhìn tổng quan cột chữ

Thói quen 5 bước có phiên bản cho văn bản: đếm rỗng → đo độ dài → soi vài mẫu ngắn/dài.

In [ ]:
# TODO: đếm comment NaN; tạo c = cột comments đã bỏ NaN; tính độ dài từng comment
so_nan = ...
c = ...
do_dai = ...            # gợi ý: .str.len()

# --- Ô kiểm tra ---
assert so_nan == 36 and len(c) == 690076
assert do_dai.median() == 115.0
print(f"{so_nan} comment rỗng; độ dài trung vị {do_dai.median():.0f} ký tự.")

In [ ]:
# TODO: đếm số comment NGẮN hơn 20 ký tự, rồi xem thử 5 comment ngắn nhất
so_ngan = ...

# --- Ô kiểm tra ---
assert so_ngan == 58987
print(f"{so_ngan:,} comment < 20 ký tự (~8.5%)")
c[do_dai < 3].head(5).tolist()

Gần 59 nghìn comment cực ngắn — nhiều cái chỉ là dấu "`.`". Với hợp phần LLM của
bài tập lớn, gửi các comment này cho model là **đốt quota vô ích**: một quy tắc lọc
độ-dài-tối-thiểu sẽ nằm trong pipeline của bạn.

### Bước 2 · Rác HTML: thẻ `<br/>`

Airbnb lưu xuống dòng thành thẻ `<br/>` ngay trong văn bản.

In [ ]:
# TODO: đếm số comment chứa "<br/>" (regex=False!), rồi tạo c_sach thay "<br/>" bằng " "
so_br = ...
c_sach = ...

# --- Ô kiểm tra ---
assert so_br == 116471
assert int(c_sach.str.contains("<br/>", regex=False).sum()) == 0
print(f"{so_br:,} comment có <br/> — sau khi thay: 0. Làm sạch xong, có kiểm chứng.")

## Phần 2 · Tín hiệu từ văn bản (~30 phút)

### Bước 3 · Heuristic ngôn ngữ

Bài giảng dựng heuristic tiếng Tây Ban Nha cho cột `name` — giờ áp cho `comments`
(bộ từ khoá chỉnh cho văn review):

In [ ]:
PAT_ES = r"ción|ñ|muy|excelente|departamento"

# TODO: tạo mặt nạ la_es (contains PAT_ES, không phân biệt hoa thường) và tính tỷ lệ
la_es = ...
ty_le_es = ...

# --- Ô kiểm tra ---
assert round(ty_le_es, 3) == 0.654
print(f"~{ty_le_es:.0%} review mang dấu hiệu tiếng Tây Ban Nha.")

In [ ]:
# TODO: so độ dài trung vị của nhóm es và nhóm còn lại
do_dai_sach = c_sach.str.len()
len_es = ...
len_khac = ...

# --- Ô kiểm tra ---
assert len_es == 118.0 and len_khac == 105.0
print(f"Trung vị độ dài — nhóm es: {len_es:.0f} · nhóm khác: {len_khac:.0f} ký tự.")

Heuristic này **thô** (một review tiếng Anh chứa từ "departamento" sẽ bị gán es) —
nhưng nó cho bức tranh nhanh: khách nói tiếng Tây Ban Nha chiếm ~2/3 và viết dài hơn một chút.
Buổi 11 sẽ đo chất lượng heuristic này bằng nhãn tay, cạnh một LLM.

### Bước 4 · str.extract trên văn bản tự do — và cái bẫy của chính bạn

Khách hay viết "cerca de X" / "near X" (gần X). Trích thử **X** ra xem khách nhắc gì:

In [ ]:
PAT_GAN = r"(?:cerca de la |cerca del |cerca de |near the |near )([A-Za-zÁ-Úá-úñÑ]+)"

# TODO: dùng str.extract với PAT_GAN (expand=False) trên c_sach; đếm số dòng trích được
gan = ...
so_trich = ...

# --- Ô kiểm tra ---
assert so_trich == 33841
gan.str.lower().value_counts().head(8)

Đọc bảng top: `metro` (7.640 lần), `centro`, `estación`, `mall`… — **và cả `todo`,
`muchos`**. Hai từ sau là *false positive*: "cerca de **todo**" nghĩa là "gần **mọi thứ**",
không phải địa điểm. Regex trích đúng mẫu chữ nhưng không hiểu nghĩa — soi `value_counts`
của chính kết quả mình trích là cách rẻ nhất để phát hiện điều đó. (Còn `movistar`?
Movistar Arena — nhà thi đấu lớn của Santiago. Tín hiệu thật!)

## Phần 3 · Bài tự làm ✅ mở (làm xong sớm / về nhà)

Được dùng AI theo quy trình 5 bước; ghi lại prompt + cách kiểm chứng.

### Tự làm 1 · Từ khoá hai ngôn ngữ

Xây 2 tín hiệu: `wifi` (mẫu `wifi|internet`) và `parking` (mẫu `parking|estacionamiento`).
Tính **tỷ lệ nhắc đến** của từng tín hiệu trong nhóm es và nhóm còn lại. Nhóm nào quan tâm
parking hơn? Thử lý giải bằng 1–2 câu (gợi ý: ai lái xe đến Santiago?).

### Tự làm 2 · Cải thiện mẫu "cerca de"

Sửa `PAT_GAN` để loại các false positive kiểu "todo"/"muchos" (gợi ý: thêm nhóm loại trừ
hoặc lọc kết quả sau khi trích bằng danh sách từ dừng). Đếm lại top 8 — bảng mới sạch hơn
bảng cũ ở chỗ nào? Ghi lại quy trình: sửa mẫu → chạy lại → so kết quả (đúng vòng lặp
"nới dần" của bài giảng).

In [ ]:
# Viết bài tự làm của bạn ở đây

## Tóm tắt buổi lab

| Bạn đã làm | Sẽ gặp lại ở |
|---|---|
| Khảo sát cột chữ: rỗng, độ dài, mẫu ngắn | lọc review trước khi gửi LLM (buổi 11) |
| Làm sạch `<br/>` + kiểm chứng bằng đếm | pipeline bài tập lớn |
| Heuristic ngôn ngữ trên 690k review | baseline cho hợp phần LLM |
| extract + tự soi false positive | kỷ luật kiểm chứng mọi tín hiệu trích |

Buổi lý thuyết tới: **dữ liệu thời gian** — 690 nghìn mốc `date` này sẽ thành chuỗi
thời gian thật sự.